> ⚠️ **Aviso sobre uso de Inteligência Artificial**
> Todas as partes deste trabalho devem ser da autoria do aluno. Qualquer uso de ferramentas generativas de IA, como ChatGPT, Claude, Copilot ou similares, é proibido. O uso de IA generativa será considerado má conduta acadêmica e estará sujeito à aplicação do código disciplinar, pois as tarefas deste trabalho foram elaboradas para desafiar o aluno a desenvolver conhecimentos de base, pensamento crítico e habilidades de resolução de problemas. O uso da tecnologia de IA limitaria sua capacidade de desenvolver essas competências e de atingir os objetivos de aprendizagem desta disciplina.

---

# Assessment — Dados e SQL

Este Assessment é resolvido **inteiramente neste notebook do Deepnote**, usando **blocos SQL nativos**. Ele tem duas partes:

- **Parte A — Queries (12 exercícios):** DDL, manipulação de dados e consultas com JOINs e agregações, sobre dois bancos distintos.
- **Parte B — Projeto de Modelagem (4 exercícios):** análise de entidades, relacionamentos, normalização e implementação física de um schema a partir de um caso de negócio. Resolvida nas células finais deste mesmo notebook.

### Como os dados entram: importação de CSV

Os dados **não** vêm de scripts `CREATE TABLE` prontos, mas em um conjunto de arquivos **CSV**. A importação cria tabelas com tipos inferidos e **sem restrições** (sem chaves, sem `NOT NULL`, sem `FOREIGN KEY`) — exatamente como dados crus chegam no mundo real. Parte dos exercícios consiste justamente em transformar esses dados crus em um schema bem projetado.

### Célula de importação = botão de reset

A célula de importação de cada banco usa `CREATE OR REPLACE TABLE`. **Reexecutá-la a qualquer momento restaura o estado inicial** dos dados. Como vários exercícios modificam tabelas (`INSERT`, `UPDATE`, `DELETE`, `DROP`), trate cada exercício como se partisse do estado recém-importado: se um exercício anterior alterou os dados, **reexecute a célula de importação do banco antes de resolver um exercício de consulta**.

### Sobre datas

Todos os filtros temporais usam datas fixas no formato `'AAAA-MM-DD'`. **Não use `CURRENT_DATE`** — os enunciados pressupõem datas explícitas para resultados reproduzíveis.

---

# 🅐 Parte A — Queries

# Banco 1 — Biblioteca Municipal

## Arquivos CSV do Banco 1 (o professor disponibiliza no projeto)

**`autores.csv`**
```csv
id,nome,nacionalidade,ano_nascimento
1,Machado de Assis,Brasileira,1839
2,Clarice Lispector,Brasileira,1920
3,Jorge Amado,Brasileira,1912
4,Gabriel García Márquez,Colombiana,1927
5,Haruki Murakami,Japonesa,1949
6,Chimamanda Adichie,Nigeriana,1977
```

**`livros.csv`**
```csv
id,titulo,autor_id,ano_publicacao,exemplares_disponiveis
1,Dom Casmurro,1,1899,3
2,Memórias Póstumas de Brás Cubas,1,1881,2
3,Quincas Borba,1,1891,1
4,A Hora da Estrela,2,1977,1
5,Água Viva,2,1973,2
6,Capitães da Areia,3,1937,4
7,"Gabriela, Cravo e Canela",3,1958,3
8,Cem Anos de Solidão,4,1967,2
9,O Amor nos Tempos do Cólera,4,1985,1
10,Norwegian Wood,5,1987,2
11,Americanah,6,2013,3
```

**`membros.csv`**
```csv
id,nome,email,data_inscricao
1,Ana Silva,ana@biblio.org,2023-01-15
2,Bruno Lima,bruno@biblio.org,2023-03-22
3,Carla Reis,carla@biblio.org,2024-02-10
4,Davi Castro,davi@biblio.org,2024-05-01
5,Edu Martins,edu@biblio.org,2024-08-12
```

**`emprestimos.csv`** (campos vazios em `data_devolucao_real` representam empréstimos não devolvidos → `NULL`)
```csv
id,livro_id,membro_id,data_emprestimo,data_devolucao_prevista,data_devolucao_real,valor_multa
1,1,1,2023-08-15,2023-08-29,2023-08-28,0
2,4,2,2023-11-10,2023-11-24,2023-12-01,5
3,6,1,2024-02-05,2024-02-19,2024-02-18,0
4,2,3,2026-02-10,2026-02-24,,15
5,8,4,2026-01-20,2026-02-03,,28
6,10,2,2026-03-01,2026-03-15,,10
7,5,5,2025-12-10,2025-12-24,2025-12-23,0
8,9,5,2026-05-01,2026-05-30,,0
9,11,4,2025-09-20,2025-10-04,2025-10-04,0
```


## Célula de importação do Banco 1 — *bloco SQL · execute antes de tudo*



> Após esta célula, as quatro tabelas existem com tipos inferidos, **mas sem chaves nem restrições** — você vai lidar com isso nos primeiros exercícios.

---


In [1]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_1 = _dntk.execute_sql(
  'CREATE OR REPLACE TABLE livros AS SELECT * FROM read_csv_auto(\'livros.csv\');',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_1

,Count
0,11


In [2]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- IMPORTAÇÃO / RESET — Banco 1: Biblioteca Municipal\nCREATE OR REPLACE TABLE autores (\n    id INTEGER PRIMARY KEY,\n    nome TEXT NOT NULL,\n    nacionalidade TEXT,\n    ano_nascimento INTEGER\n);\n\nINSERT INTO autores SELECT * FROM read_csv_auto(\'autores.csv\');CREATE OR REPLACE TABLE livros      AS SELECT * FROM read_csv_auto(\'livros.csv\');\nCREATE OR REPLACE TABLE membros     AS SELECT * FROM read_csv_auto(\'membros.csv\');\nCREATE OR REPLACE TABLE emprestimos AS SELECT * FROM read_csv_auto(\'emprestimos.csv\');\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,Count
0,9


### Exercício 1

A biblioteca decidiu cadastrar formalmente seus funcionários numa tabela **bem definida desde o início** (diferente das tabelas importadas do CSV).

**Tarefa.** Crie uma tabela `funcionarios` com restrições adequadas:

- `id`
- `nome`
- `email`
- `cargo`
- `data_admissao`

Insira os 4 funcionários abaixo e execute um `SELECT` ordenado por `data_admissao`.

| id | nome | email | cargo | data_admissao |
|---|---|---|---|---|
| 1 | Lúcia Andrade | lucia@biblio.org | Bibliotecária | 2022-04-10 |
| 2 | Renato Souza | renato@biblio.org | Atendente | 2023-09-01 |
| 3 | Marta Oliveira | marta@biblio.org | Bibliotecária | 2024-01-15 |
| 4 | Iago Pereira | iago@biblio.org | Auxiliar | 2024-11-20 |


---


In [3]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 1\n-- Escreva sua solução SQL aqui.\n\nCREATE TABLE funcionarios (\n    id INT PRIMARY KEY,\n    nome VARCHAR(100) NOT NULL,\n    email VARCHAR(100) UNIQUE NOT NULL,\n    cargo VARCHAR(50) NOT NULL,\n    data_admissao DATE NOT NULL\n);\n\nINSERT INTO funcionarios VALUES\n(1, \'Lúcia Andrade\', \'lucia@biblio.org\', \'Bibliotecária\', \'2022-04-10\'),\n(2, \'Renato Souza\', \'renato@biblio.org\', \'Atendente\', \'2023-09-01\'),\n(3, \'Marta Oliveira\', \'marta@biblio.org\', \'Bibliotecária\', \'2024-01-15\'),\n(4, \'Iago Pereira\', \'iago@biblio.org\', \'Auxiliar\', \'2024-11-20\');\n\nSELECT * FROM funcionarios ORDER BY data_admissao;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,id,nome,email,cargo,data_admissao
0,1,Lúcia Andrade,lucia@biblio.org,Bibliotecária,2022-04-10
1,2,Renato Souza,renato@biblio.org,Atendente,2023-09-01
2,3,Marta Oliveira,marta@biblio.org,Bibliotecária,2024-01-15
3,4,Iago Pereira,iago@biblio.org,Auxiliar,2024-11-20


# 

### Exercício 2

A tabela `livros` veio do CSV **sem nenhuma restrição**: não há chave primária, nada impede um título nulo, e nada garante que `autor_id` aponte para um autor real. Você vai criar uma versão bem modelada e migrar os dados.

**Tarefa.**
1. Crie uma tabela `livros_catalogo` com: `id`; `titulo`; `autor_id` como `FOREIGN KEY` referenciando `autores(id)`; `ano_publicacao`; `exemplares_disponiveis`.
2. Migre **todos** os registros de `livros` para `livros_catalogo`.


---


In [4]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 2\n-- Escreva sua solução SQL aqui.\n\nCREATE TABLE livros_catalogo (\n    id INT PRIMARY KEY,\n    titulo TEXT NOT NULL,\n    autor_id INT NOT NULL,\n    ano_publicacao INT,\n    exemplares_disponiveis INT,\n\n    FOREIGN KEY (autor_id) REFERENCES autores(id)\n);\n\nINSERT INTO livros_catalogo (id, titulo, autor_id, ano_publicacao, exemplares_disponiveis)\nSELECT id, titulo, autor_id, ano_publicacao, exemplares_disponiveis\nFROM livros;\n\nSELECT * FROM livros_catalogo;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,id,titulo,autor_id,ano_publicacao,exemplares_disponiveis
0,1,Dom Casmurro,1,1899,3
1,2,Memórias Póstumas de Brás Cubas,1,1881,2
2,3,Quincas Borba,1,1891,1
3,4,A Hora da Estrela,2,1977,1
4,5,Água Viva,2,1973,2
5,6,Capitães da Areia,3,1937,4
6,7,"Gabriela, Cravo e Canela",3,1958,3
7,8,Cem Anos de Solidão,4,1967,2
8,9,O Amor nos Tempos do Cólera,4,1985,1
9,10,Norwegian Wood,5,1987,2


### Exercício 3

A coordenação pediu uma listagem do acervo identificando quem escreveu cada livro.

**Tarefa.** Escreva uma única query com `INNER JOIN` entre `livros` e `autores` retornando `titulo`, `nome` do autor e `ano_publicacao`. Ordene por nome do autor (A→Z) e depois por ano de publicação (mais antigo → mais recente).


---


In [5]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 3\n-- Escreva sua solução SQL aqui.\n\nSELECT l.titulo, a.nome AS autor, l.ano_publicacao\nFROM livros l\nJOIN autores a ON l.autor_id = a.id\nORDER BY autor, l.ano_publicacao;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,titulo,autor,ano_publicacao
0,Americanah,Chimamanda Adichie,2013
1,Água Viva,Clarice Lispector,1973
2,A Hora da Estrela,Clarice Lispector,1977
3,Cem Anos de Solidão,Gabriel García Márquez,1967
4,O Amor nos Tempos do Cólera,Gabriel García Márquez,1985
5,Norwegian Wood,Haruki Murakami,1987
6,Capitães da Areia,Jorge Amado,1937
7,"Gabriela, Cravo e Canela",Jorge Amado,1958
8,Memórias Póstumas de Brás Cubas,Machado de Assis,1881
9,Quincas Borba,Machado de Assis,1891


### Exercício 4

A biblioteca quer identificar **livros que nunca foram emprestados** (candidatos a desativação do acervo).

**Tarefa.** Use a técnica de *anti-join*: `LEFT JOIN` entre `livros` e `emprestimos`, mantendo apenas as linhas em que **não há empréstimo correspondente**. Retorne `titulo` e `nome` do autor de cada livro nunca emprestado. 


---


In [6]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 4\n-- Escreva sua solução SQL aqui.\n\nSELECT l.titulo, a.nome AS autor\nFROM livros l\nJOIN autores a ON l.autor_id = a.id\nLEFT JOIN emprestimos e ON l.id = e.id\nWHERE e.id IS NULL',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,titulo,autor
0,Norwegian Wood,Haruki Murakami
1,Americanah,Chimamanda Adichie


### Exercício 5 · ⚠️ modifica dados

A biblioteca vai registrar o **gênero literário** dos livros e cadastrar **duas aquisições novas**.

**Tarefa.**
1. `ALTER TABLE livros` adicionando uma coluna de texto chamada `genero` .
2. Com `UPDATE` e `WHERE`, defina: `'Dom Casmurro'` → `'Romance'`; `'A Hora da Estrela'` → `'Romance'`; `'Capitães da Areia'` → `'Romance Social'`.
3. **Insira dois livros novos** na tabela `livros` (ids 12 e 13), de autores já existentes, já preenchendo o `genero`:
   - id 12, `'A Paixão Segundo G.H.'`, autor_id 2, ano 1964, 2 exemplares, gênero `'Romance'`.
   - id 13, `'Kafka à Beira-Mar'`, autor_id 5, ano 2002, 1 exemplar, gênero `'Fantasia'`.
4. `SELECT` retornando `titulo` e `genero` de todos os livros com gênero preenchido.

---


In [7]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 5\n-- Escreva sua solução SQL aqui.\nALTER TABLE livros ADD COLUMN genero TEXT;\n\nUPDATE livros SET genero = \'Romance\' WHERE titulo = \'Dom Casmuro\';\nUPDATE livros SET genero = \'Romance\' WHERE titulo = \'A hora da Estrela\';\nUPDATE livros SET genero = \'Romance\' WHERE titulo = \'Capitães de areia\';\n\nINSERT INTO livros VALUES\n(12, \'A Paixão Segundo G.H.\', 2, 1964, 2, \'Romance\'),\n(13, \'Kafka à Beira-Mar\', 5, 2002, 1, \'Fantasia\');\n\nSELECT titulo, genero FROM livros WHERE genero IS NOT NULL;\n\n\n\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,titulo,genero
0,A Paixão Segundo G.H.,Romance
1,Kafka à Beira-Mar,Fantasia


### Exercício 6· ⚠️ modifica dados

A direção decidiu **reajustar as multas em 50%** para empréstimos ainda em aberto e que já têm multa.

**Tarefa.** Uma única query `UPDATE` que multiplique `valor_multa` por 1.5 apenas nas linhas em que `data_devolucao_real` é nula **e** `valor_multa` é maior que zero. Depois, `SELECT` retornando `id`, `livro_id`, `membro_id`, `valor_multa` dessas linhas.

**Saída.** Afeta 3 linhas:

| id | livro_id | membro_id | valor_multa |
|---|---|---|---|
| 4 | 2 | 3 | 22.5 |
| 5 | 8 | 4 | 42.0 |
| 6 | 10 | 2 | 15.0 |

---


In [8]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 6\n-- Escreva sua solução SQL aqui.\n\nUPDATE emprestimos \nSET valor_multa = valor_multa * 1.5\nWHERE data_devolucao_real IS NULL\nAND valor_multa > 0;\nSELECT id,livro_id, membro_id, valor_multa\nFROM emprestimos\nWHERE data_devolucao_real IS NULL\nAND valor_multa >0;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,id,livro_id,membro_id,valor_multa
0,4,2,3,23
1,5,8,4,42
2,6,10,2,15


# Banco 2 — Liga de Futebol

## Arquivos CSV do Banco 2 (o professor disponibiliza no projeto)

**`times.csv`**
```csv
id,nome
1,Flamengo
2,Palmeiras
3,Corinthians
4,Grêmio
```

**`jogadores.csv`**
```csv
id,nome,time_id
1,Pedro,1
2,Arrascaeta,1
3,Gabigol,1
4,Endrick,2
5,Veiga,2
6,Rony,2
7,Yuri Alberto,3
8,Cássio,3
9,Renato Augusto,3
10,Suárez,4
11,Diego Costa,4
12,Pepê,4
```

**`partidas.csv`**
```csv
id,time_mandante_id,time_visitante_id,gols_mandante,gols_visitante,data
1,1,2,2,1,2026-03-01
2,3,4,1,1,2026-03-08
3,1,3,3,0,2026-03-15
4,2,4,2,2,2026-03-22
5,2,3,1,0,2026-03-29
6,1,4,0,2,2026-04-05
7,1,2,3,1,2026-04-12
8,1,3,1,0,2026-04-19
9,4,3,2,1,2026-04-26
10,4,2,1,0,2026-05-03
```

**`gols.csv`**
```csv
id,partida_id,jogador_id,minuto
1,1,1,12
2,1,1,67
3,1,4,38
4,2,7,22
5,2,10,80
6,3,3,15
7,3,3,55
8,3,2,71
9,4,5,18
10,4,5,60
11,4,11,33
12,4,11,88
13,5,6,25
14,6,10,41
15,6,11,76
16,7,1,9
17,7,3,44
18,7,2,73
19,7,4,81
20,8,1,52
21,9,10,19
22,9,10,66
23,9,7,30
24,10,11,28
```


## Célula de importação do Banco 2 — *bloco SQL · execute antes dos exercícios 7 a 12*



---


In [9]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- IMPORTAÇÃO / RESET — Banco 2: Liga de Futebol\nCREATE OR REPLACE TABLE times     AS SELECT * FROM read_csv_auto(\'times.csv\');\nCREATE OR REPLACE TABLE jogadores AS SELECT * FROM read_csv_auto(\'jogadores.csv\');\nCREATE OR REPLACE TABLE partidas  AS SELECT * FROM read_csv_auto(\'partidas.csv\');\nCREATE OR REPLACE TABLE gols      AS SELECT * FROM read_csv_auto(\'gols.csv\');\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,Count
0,24


### Exercício 7

A redação esportiva quer um relatório detalhado de todos os gols, mostrando quem marcou e os dois times de cada partida.

**Tarefa.** Uma única query com o join adequado entre as tabelas necessárias retornando, por gol: nome do jogador, nome do time do jogador, nome do time mandante, nome do time visitante e minuto. Use **alias** na tabela `times` para referenciá-la três vezes (time do jogador, mandante, visitante). Ordene por `data` e depois por `minuto`.

---


In [10]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 7\n-- Escreva sua solução SQL aqui.\n\nSELECT \nj.nome as jogador,\ntj.nome AS time_jogador,\ntm.nome AS time_jogador,\ntv.nome AS time_visitante,\ng.minuto,\nFROM gols g \nJOIN jogadores j ON g.jogador_id = j.id\nJOIN times tj ON j.time_id = tj.id\nJOIN partidas p ON g.partida_id = p.id\nJOIN times tm ON p.time_mandante_id = tm.id\nJOIN times tv ON p.time_visitante_id = tv.id\nORDER BY p.data, g.minuto;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,jogador,time_jogador,time_jogador_1,time_visitante,minuto
0,Pedro,Flamengo,Flamengo,Palmeiras,12
1,Endrick,Palmeiras,Flamengo,Palmeiras,38
2,Pedro,Flamengo,Flamengo,Palmeiras,67
3,Yuri Alberto,Corinthians,Corinthians,Grêmio,22
4,Suárez,Grêmio,Corinthians,Grêmio,80
5,Gabigol,Flamengo,Flamengo,Corinthians,15
6,Gabigol,Flamengo,Flamengo,Corinthians,55
7,Arrascaeta,Flamengo,Flamengo,Corinthians,71
8,Veiga,Palmeiras,Palmeiras,Grêmio,18
9,Diego Costa,Grêmio,Palmeiras,Grêmio,33


### Exercício 8

A comissão técnica quer um relatório **completo** dos jogadores, incluindo os que ainda não marcaram.

**Tarefa.** o join adequado entre as tabelas necessárias, agrupando por jogador e contando gols, incluindo **todos os 12 jogadores** (0 para quem não marcou). Ordene por total de gols decrescente, desempate por nome. 

---


In [11]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{"sortBy":[],"filters":[],"pageSize":25,"pageIndex":0,"columnOrder":["nome","total_gols"],"hiddenColumnIds":[],"columnDisplayNames":[],"conditionalFilters":[],"cellFormattingRules":[],"wrappedTextColumnIds":[]}')
else:
  _deepnote_current_table_attrs = '{"sortBy":[],"filters":[],"pageSize":25,"pageIndex":0,"columnOrder":["nome","total_gols"],"hiddenColumnIds":[],"columnDisplayNames":[],"conditionalFilters":[],"cellFormattingRules":[],"wrappedTextColumnIds":[]}'

_dntk.execute_sql(
  '-- Exercício 8\n-- Escreva sua solução SQL aqui.\n\nSELECT j.nome, COUNT (g.id) AS total_gols\nFROM jogadores j \nLEFT JOIN gols g ON j.id = g.jogador_id\nGROUP BY j.id, j.nome \nORDER BY total_gols DESC, j.nome ASC\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,nome,total_gols
0,Diego Costa,4
1,Pedro,4
2,Suárez,4
3,Gabigol,3
4,Arrascaeta,2
5,Endrick,2
6,Veiga,2
7,Yuri Alberto,2
8,Rony,1
9,Cássio,0


### Exercício 9

A organização quer mapear **duplas de companheiros de time** para uma campanha de marketing.

**Tarefa.** Use o join adequado para juntas a tabela `jogadores` com ela mesma e formar pares de jogadores do **mesmo time**. Evite duplicatas e auto-pares. Retorne nome do time, nome do jogador 1 e nome do jogador 2, ordenado por time.

---


In [12]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 9\n-- Escreva sua solução SQL aqui.\n\nSELECT \n    t.nome AS time,\n    j1.nome AS jogador_1,\n    j2.nome AS jogador_2\nFROM jogadores j1\nJOIN jogadores j2 \n    ON j1.time_id = j2.time_id\nJOIN times t \n    ON j1.time_id = t.id\nWHERE j1.id < j2.id\nORDER BY t.nome;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,time,jogador_1,jogador_2
0,Corinthians,Cássio,Renato Augusto
1,Corinthians,Yuri Alberto,Cássio
2,Corinthians,Yuri Alberto,Renato Augusto
3,Flamengo,Arrascaeta,Gabigol
4,Flamengo,Pedro,Arrascaeta
5,Flamengo,Pedro,Gabigol
6,Grêmio,Diego Costa,Pepê
7,Grêmio,Suárez,Diego Costa
8,Grêmio,Suárez,Pepê
9,Palmeiras,Veiga,Rony


### Exercício 10

Antes da fase eliminatória, a comissão quer o **saldo de gols** de cada time (gols pró menos gols contra), considerando todas as partidas.

**Tarefa.** Cada time aparece em partidas tanto como mandante quanto como visitante. Una `times` com `partidas` e use calcule:
- **gols pró** — `gols_mandante` quando o time é mandante; `gols_visitante` quando é visitante;
- **gols contra** — o inverso;
- **saldo** — pró menos contra.

Retorne `nome`, `gols_pro`, `gols_contra`, `saldo`, ordenado por saldo decrescente.

**Saída.** 4 linhas:

| nome | gols_pro | gols_contra | saldo |
|---|---|---|---|
| Flamengo | 9 | 4 | 5 |
| Grêmio | 8 | 4 | 4 |
| Palmeiras | 5 | 8 | -3 |
| Corinthians | 2 | 8 | -6 |

---


In [13]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 10\n-- Escreva sua solução SQL aqui.\n\nSELECT \n    nome,\n    SUM(gols_feitos) AS gols_pro,\n    SUM(gols_sofridos) AS gols_contra,\n    SUM(gols_feitos) - SUM(gols_sofridos) AS saldo\nFROM (\n    SELECT t.nome, p.gols_mandante AS gols_feitos, p.gols_visitante AS gols_sofridos\n    FROM times t\n    JOIN partidas p ON t.id = p.time_mandante_id\n    \n    UNION ALL\n    \n    SELECT t.nome, p.gols_visitante AS gols_feitos, p.gols_mandante AS gols_sofridos\n    FROM times t\n    JOIN partidas p ON t.id = p.time_visitante_id\n) AS tabela_base\nGROUP BY nome\nORDER BY saldo DESC;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,nome,gols_pro,gols_contra,saldo
0,Flamengo,9.0,4.0,5.0
1,Grêmio,8.0,4.0,4.0
2,Palmeiras,5.0,8.0,-3.0
3,Corinthians,2.0,8.0,-6.0


### Exercício 11

A query abaixo deveria retornar os **artilheiros da temporada** (jogadores com mais de 1 gol) com o nome do time, mas está com defeitos.

**Query quebrada (ponto de partida)**

```sql
SELECT j.nome, COUNT(*) AS gols, t.nome
FROM jogadores j
LEFT JOIN gols g ON g.jogador_id = j.id
JOIN times t ON j.time_id = t.id
WHERE COUNT(*) > 1
GROUP BY j.nome
ORDER BY gols;
```

**Tarefa.** Identifique **todos** os defeitos, corrija a query e produza uma versão funcional que retorne nome do jogador, nome do time e total de gols, apenas para jogadores com mais de 1 gol, ordenado do maior para o menor. Em comentários SQL no início, explique cada defeito e por que sua correção resolve.

---


In [14]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 11\n-- Escreva sua solução SQL aqui.\n\n-- Defeito 1: O COUNT estava no WHERE. Correção: Mudei para HAVING porque WHERE não aceita função de agregação.\n-- Defeito 2: Faltava o t.nome no GROUP BY. Correção: Adicionei t.nome junto com j.nome para o banco não dar erro de coluna não agrupada.\n-- Defeito 3: A ordenação padrão é crescente. Correção: Adicionei o DESC no ORDER BY para mostrar do maior para o menor.\n\nSELECT \n    j.nome,\n    t.nome AS time,\n    COUNT(g.id) AS gols\nFROM jogadores j\nLEFT JOIN gols g ON g.jogador_id = j.id\nJOIN times t ON j.time_id = t.id\nGROUP BY j.nome, t.nome\nHAVING COUNT(g.id) > 1\nORDER BY gols DESC;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,nome,time,gols
0,Pedro,Flamengo,4
1,Suárez,Grêmio,4
2,Diego Costa,Grêmio,4
3,Gabigol,Flamengo,3
4,Endrick,Palmeiras,2
5,Yuri Alberto,Corinthians,2
6,Arrascaeta,Flamengo,2
7,Veiga,Palmeiras,2


### Exercício 12 · ⚠️ modifica dados

Uma nova rodada foi disputada e precisa ser registrada no banco.

**Tarefa.**
1. **Insira** em `partidas` a partida de id 11: mandante Palmeiras (2), visitante Corinthians (3), 2×1, data `'2026-05-10'`.
2. **Insira** em `gols` os três gols dessa partida: jogador 5 (Veiga) aos 20', jogador 4 (Endrick) aos 55', jogador 7 (Yuri Alberto) aos 70'. Use ids de gol 25, 26 e 27.
3. Escreva uma query que confirme a inserção: liste os gols da partida 11 com nome do jogador e minuto (`INNER JOIN` entre `gols` e `jogadores`, filtrando `partida_id = 11`).

**Saída.** 3 linhas: Veiga (20), Endrick (55), Yuri Alberto (70). Demonstra que os novos dados foram inseridos corretamente.

---


In [15]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 12\n-- Escreva sua solução SQL aqui.\n\n-- Inserindo a nova partida\nINSERT INTO partidas \nVALUES (11, 2, 3, 2, 1, \'2026-05-10\');\n\n-- Inserindo os gols da partida 11\nINSERT INTO gols VALUES (25, 11, 5, 20);\nINSERT INTO gols VALUES (26, 11, 4, 55);\nINSERT INTO gols VALUES (27, 11, 7, 70);\n\n-- Conferindo os gols da partida\nSELECT \n    j.nome,\n    g.minuto\nFROM gols g\nJOIN jogadores j ON g.jogador_id = j.id\nWHERE g.partida_id = 11;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,nome,minuto
0,Endrick,55
1,Veiga,20
2,Yuri Alberto,70


# 🅑 Parte B — Projeto de Modelagem · Marketplace Artesanal

Esta parte é resolvida **nas células finais deste notebook**. Ela mistura **células de texto (Markdown)** — onde você escreve suas análises — e **células SQL** — onde você implementa e testa o schema. 

Diferente da Parte A (foco em execução), aqui o foco é **projetar**. Leia o cenário, importe a planilha desnormalizada para inspecioná-la, e construa um modelo relacional correto.

## A planilha atual — arquivo CSV (o professor disponibiliza no projeto)

**`pedidos_planilha.csv`** (uma linha por item de pedido; dados de cliente e pedido repetem entre linhas; a coluna `categorias` empacota vários valores separados por `;`)
```csv
pedido_id,data,cliente,cli_email,cli_cidade,vendedor,vend_email,vend_pix,produto,preco_unit,qtd,categorias
P001,2026-04-10,Marina Costa,marina@mail.com,São Paulo,Atelier Lua,lua@atelier.com.br,lua-pix-01,Vaso de cerâmica azul,85.00,2,"decoração; cerâmica"
P001,2026-04-10,Marina Costa,marina@mail.com,São Paulo,Tecidos Nelm,nelm@tecidos.com.br,nelm-pix-02,Almofada bordada flores,45.00,1,"casa; têxtil; decoração"
P002,2026-04-11,Tiago Reis,tiago@mail.com,Curitiba,Atelier Lua,lua@atelier.com.br,lua-pix-01,Vaso de cerâmica azul,85.00,1,"decoração; cerâmica"
P003,2026-04-12,Marina Costa,marina@mail.com,São Paulo,Madeira Viva,madv@artesa.com.br,madv-pix-03,Porta-copos rústico (kit 4),60.00,1,"casa; madeira"
P003,2026-04-12,Marina Costa,marina@mail.com,São Paulo,Tecidos Nelm,nelm@tecidos.com.br,nelm-pix-02,Almofada bordada flores,45.00,2,"casa; têxtil; decoração"
P004,2026-04-15,Larissa Veiga,lari@mail.com,Recife,Madeira Viva,madv@artesa.com.br,madv-pix-03,Tábua de queijo entalhada,110.00,1,"cozinha; madeira"
```


## Célula de importação da planilha — *bloco SQL*


In [16]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- IMPORTAÇÃO — Planilha denormalizada do Marketplace\nCREATE OR REPLACE TABLE pedidos_planilha AS SELECT * FROM read_csv_auto(\'pedidos_planilha.csv\');\nSELECT * FROM pedidos_planilha;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,pedido_id,data,cliente,cli_email,cli_cidade,vendedor,vend_email,vend_pix,produto,preco_unit,qtd,categorias
0,P001,2026-04-10,Marina Costa,marina@mail.com,São Paulo,Atelier Lua,lua@atelier.com.br,lua-pix-01,Vaso de cerâmica azul,85.0,2,decoração; cerâmica
1,P001,2026-04-10,Marina Costa,marina@mail.com,São Paulo,Tecidos Nelm,nelm@tecidos.com.br,nelm-pix-02,Almofada bordada flores,45.0,1,casa; têxtil; decoração
2,P002,2026-04-11,Tiago Reis,tiago@mail.com,Curitiba,Atelier Lua,lua@atelier.com.br,lua-pix-01,Vaso de cerâmica azul,85.0,1,decoração; cerâmica
3,P003,2026-04-12,Marina Costa,marina@mail.com,São Paulo,Madeira Viva,madv@artesa.com.br,madv-pix-03,Porta-copos rústico (kit 4),60.0,1,casa; madeira
4,P003,2026-04-12,Marina Costa,marina@mail.com,São Paulo,Tecidos Nelm,nelm@tecidos.com.br,nelm-pix-02,Almofada bordada flores,45.0,2,casa; têxtil; decoração
5,P004,2026-04-15,Larissa Veiga,lari@mail.com,Recife,Madeira Viva,madv@artesa.com.br,madv-pix-03,Tábua de queijo entalhada,110.0,1,cozinha; madeira


## Cenário

A startup **Feito à Mão Marketplace** controla seus pedidos na planilha acima. Com o crescimento, ela virou uma bagunça: dados duplicados, células com múltiplos valores e dificuldade para consultar quem vendeu o quê. Você foi contratado para projetar a base relacional que vai substituí-la.

## Regras de negócio (cada uma implica uma decisão de modelagem)

1. **Um cliente pode fazer vários pedidos**, mas cada pedido pertence a um único cliente.
2. **Um pedido pode conter itens de vendedores diferentes** (veja P001 e P003).
3. **Um produto pertence a um único vendedor** — não há produtos compartilhados.
4. **Um produto pode estar em uma ou mais categorias**, e uma categoria pode ter vários produtos.
5. **Cada vendedor possui uma única conta bancária para recebimentos** (com a chave Pix), e essa conta pertence exclusivamente a ele.
6. **O email de cliente e de vendedor identifica a pessoa de forma única** — não pode haver dois cadastros com o mesmo email.
7. **O preço de um produto pode mudar com o tempo**, mas o pedido deve preservar o preço cobrado **no momento da compra**.
8. **Categorias são poucas e padronizadas** — devem viver numa tabela própria, não como texto solto.

---


## ✍️ Exercício 13 — Reflexão e análise de entidades

Escreva nesta célula:

- **Reflexão inicial (~10 linhas):** por que um banco relacional é apropriado aqui? Que problemas concretos a planilha já mostra que um bom modelo resolve? *(Cobre a base histórica/conceitual: por que relacional, vantagens de boas práticas.)*
- **Entidades:** liste cada entidade identificada (ex: Cliente, Vendedor, Conta Bancária, Produto, Categoria, Pedido, Item de Pedido), com seus atributos e a justificativa de por que é entidade própria — e não atributo de outra.

---


## Resposta — Exercício 13

Um banco relacional seria melhor para esse caso, porque a planilha começa a ficar bagunçada conforme ela vai crescendo. Além de ter muita informação repitida, tipo nome do cliente e o vendedor aparece várias vezes, se precisar mudar algum dado pode dar erro. A parte das categorias estão todas juntas numa célula só, vai dificultar na hora de filtrar ou fazer consulta. 

O ideal é separar em tabelas porque fica mais organizado e evitar de duplicar os dados, se fizer isso cada coisa vai ficar no se lugar, tipo cliente,pedido, produto, vendedor, e depois só liga eles com ID.

ENTIDADES: 
cliente: Guarda os dados de quem compra(ID,Nome,email,cidade).

vendedor: quem vende os produtos(id,nome e email)

conta_bancaria: guarda o pix do vendedor, porque cada vendedor tem um só.

produto: os itens vendidos (id,nome, preco,vendedor_id)

categorias: tipos de produtos,porque um produto pode ter várias categorias

pedido: compra feita pelo cliente (id,data.cliente_id)

Produto_categoria: liga o produto com a categoria, porque é muitos, para muitos.




## ✍️ Exercício 14 — Relacionamentos e chaves 
Escreva nesta célula um diagrama textual de relacionamentos. Para cada relação entre entidades, indique: entidades envolvidas, cardinalidade (1:1, 1:N, N:N) e como será implementada (chave estrangeira em qual lado, ou tabela de ligação). Garanta cobrir explicitamente:

- o **1:1** entre vendedor e conta bancária;
- os **1:N** (vendedor → produto, cliente → pedido, pedido → item de pedido);
- o **N:N** entre produto e categoria;
- onde o **preço do momento da compra** é registrado (regra 7).

---


## Resposta — Exercício 14

Cliente → Pedido (1:N)
Um cliente pode fazer vários pedidos. A chave fica em pedido (cliente_id).

Vendedor → Conta_Bancaria (1:1)
Cada vendedor tem uma conta bancária só. A chave fica em conta_bancaria (vendedor_id).

Vendedor → Produto (1:N)
Um vendedor pode ter vários produtos. A chave fica em produto (vendedor_id).

Produto → Categoria (N:N)
Um produto pode ter várias categorias e categoria pode ter vários produtos. Isso é feito com a tabela produto_categoria (produto_id, categoria_id).

Pedido → Item_Pedido (1:N)
Um pedido pode ter vários itens. A chave fica em item_pedido (pedido_id).

Produto → Item_Pedido (1:N)
Um produto pode aparecer em vários itens de pedidos diferentes. A chave fica em item_pedido (produto_id).





## ✍️ Exercício 15 — Análise de normalização *(célula de texto / Markdown)*

Escreva nesta célula. Aponte **pelo menos uma violação** da planilha original em cada nível e mostre como seu modelo resolve:

- **1FN** — qual coluna guarda múltiplos valores numa só célula?
- **2FN** — qual atributo depende só de parte de uma chave composta?
- **3FN** — qual atributo depende de outro atributo não-chave (dependência transitiva)?

Cite as colunas concretas da planilha em cada caso.

---


## Resposta — Exercício 15

1FN (Valores repetidos na mesma célula): A coluna categorias quebra totalmente a primeira forma normal. No pedido P001, o vaso de cerâmica está com "decoração; cerâmica" na mesma linha. Para resolver isso, criei a tabela intermediária produto_categoria, deixando cada relação em uma linha única.

2FN (Dependência parcial de chave composta): Na planilha, se pensarmos em uma chave que identifica a linha do item (como pedido_id + produto), colunas como vendedor, vend_email e preco_unit do cadastro dependem apenas do produto em si, e não do pedido inteiro. No meu modelo, limpei isso deixando os dados do produto na tabela de produtos.

3FN (Dependência transitiva entre colunas que não são chaves): A coluna vend_pix depende diretamente do vendedor, que porque está jogado dentro da linha do pedido. A chave Pix só está ali por causa do vendedor, e não por causa do ID do pedido. Resolvi isso criando a tabela de conta_bancaria apontando para o ID do vendedor.


## 💾 Exercício 16 — Implementação física *(células SQL)*

Nas células SQL finais, implemente seu modelo:

1. Todos os `CREATE TABLE` do modelo final, com tipos, `PRIMARY KEY`, `FOREIGN KEY`, e `UNIQUE` / `NOT NULL` conforme as regras de negócio (atenção especial ao `UNIQUE` dos emails e ao lado 1:1).
2. `INSERT` reproduzindo os **6 registros da planilha** no formato normalizado. Se algum dado da planilha não tem onde caber no seu modelo, o modelo está incompleto.
3. Uma query final de verificação à sua escolha (por exemplo, um `JOIN` que reconstrói uma linha original da planilha a partir das tabelas normalizadas), demonstrando que nenhuma informação foi perdida.

> Como estas são células SQL executáveis, seu schema **roda de verdade**: se um `CREATE TABLE` ou `INSERT` tiver erro, o Deepnote vai acusar. Aproveite isso para validar antes de entregar.

---


In [17]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 16 — CREATE TABLE do modelo final\n-- Escreva aqui os comandos CREATE TABLE com PK, FK, UNIQUE e NOT NULL.\n\nCREATE TABLE clientes (\n    id INT PRIMARY KEY,\n    nome VARCHAR(100) NOT NULL,\n    email VARCHAR(100) UNIQUE NOT NULL,\n    cidade VARCHAR(50) NOT NULL\n);\n\nCREATE TABLE vendedores (\n    id INT PRIMARY KEY,\n    nome VARCHAR(100) NOT NULL,\n    email VARCHAR(100) UNIQUE NOT NULL\n);\n\nCREATE TABLE conta_bancaria (\n    id INT PRIMARY KEY,\n    vendedor_id INT UNIQUE,\n    chave_pix VARCHAR(100) NOT NULL\n);\n\nCREATE TABLE produtos (\n    id INT PRIMARY KEY,\n    nome VARCHAR(100) NOT NULL,\n    preco_atual DECIMAL(10,2) NOT NULL,\n    vendedor_id INT\n);\n\nCREATE TABLE categorias (\n    id INT PRIMARY KEY,\n    nome VARCHAR(50) NOT NULL\n);\n\nCREATE TABLE produto_categoria (\n    produto_id INT,\n    categoria_id INT\n);\n\nCREATE TABLE pedidos (\n    id VARCHAR(10) PRIMARY KEY,\n    data DATE NOT NULL,\n    cliente_id INT\n);\n\nCREATE TABLE item_pedido (\n    id INT PRIMARY KEY,\n    pedido_id VARCHAR(10),\n    produto_id INT,\n    quantidade INT,\n    preco_unitario DECIMAL(10,2)\n);\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,Count


In [18]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '\n-- Exercício 16 — INSERT dos dados normalizados\n-- Insira aqui os dados que reproduzem os 6 registros da planilha original.\n\n-- Clientes\nINSERT INTO clientes (id, nome, email, cidade) VALUES \n(1, \'Marina Costa\', \'marina@mail.com\', \'São Paulo\'),\n(2, \'Tiago Reis\', \'tiago@mail.com\', \'Curitiba\'),\n(3, \'Larissa Veiga\', \'lari@mail.com\', \'Recife\');\n\n-- Vendedores\nINSERT INTO vendedores (id, nome, email) VALUES \n(1, \'Atelier Lua\', \'lua@atelier.com.br\'),\n(2, \'Tecidos Nelm\', \'nelm@tecidos.com.br\'),\n(3, \'Madeira Viva\', \'madv@artesa.com.br\');\n\n-- Conta Bancária\nINSERT INTO conta_bancaria (id, vendedor_id, chave_pix) VALUES \n(1, 1, \'lua-pix-01\'),\n(2, 2, \'nelm-pix-02\'),\n(3, 3, \'madv-pix-03\');\n\n-- Produtos (usando preco_atual)\nINSERT INTO produtos (id, nome, preco_atual, vendedor_id) VALUES \n(1, \'Vaso de cerâmica azul\', 85.00, 1),\n(2, \'Almofada bordada flores\', 45.00, 2),\n(3, \'Porta-copos rústico (kit 4)\', 60.00, 3),\n(4, \'Tábua de queijo entalhada\', 110.00, 3);\n\n-- Categorias\nINSERT INTO categorias (id, nome) VALUES \n(1, \'decoração\'), (2, \'cerâmica\'), (3, \'casa\'), (4, \'têxtil\'), (5, \'madeira\'), (6, \'cozinha\');\n\n-- Produto_Categoria\nINSERT INTO produto_categoria (produto_id, categoria_id) VALUES \n(1,1), (1,2), (2,3), (2,4), (2,1), (3,3), (3,5), (4,6), (4,5);\n\n-- Pedidos\nINSERT INTO pedidos (id, data, cliente_id) VALUES \n(\'P001\', \'2026-04-10\', 1),\n(\'P002\', \'2026-04-11\', 2),\n(\'P003\', \'2026-04-12\', 1),\n(\'P004\', \'2026-04-15\', 3);\n\n-- Item_Pedido (com ID próprio e preco_unitario)\nINSERT INTO item_pedido (id, pedido_id, produto_id, quantidade, preco_unitario) VALUES \n(1, \'P001\', 1, 2, 85.00),\n(2, \'P001\', 2, 1, 45.00),\n(3, \'P002\', 1, 1, 85.00),\n(4, \'P003\', 3, 1, 60.00),\n(5, \'P003\', 2, 2, 45.00),\n(6, \'P004\', 4, 1, 110.00);',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,Count
0,6


In [19]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 16 — Query final de verificação\n-- Escreva aqui um JOIN que demonstre que nenhuma informação foi perdida.\n\nSELECT \n    p.id AS pedido_id,\n    p.data,\n    c.nome AS cliente,\n    v.nome AS vendedor,\n    prod.nome AS produto,\n    ip.preco_unitario AS preco_unit,\n    ip.quantidade AS qtd\nFROM pedidos p\nJOIN clientes c ON p.cliente_id = c.id\nJOIN item_pedido ip ON p.id = ip.pedido_id\nJOIN produtos prod ON ip.produto_id = prod.id\nJOIN vendedores v ON prod.vendedor_id = v.id\nORDER BY p.id;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,pedido_id,data,cliente,vendedor,produto,preco_unit,qtd
0,P001,2026-04-10,Marina Costa,Atelier Lua,Vaso de cerâmica azul,85.0,2
1,P001,2026-04-10,Marina Costa,Tecidos Nelm,Almofada bordada flores,45.0,1
2,P002,2026-04-11,Tiago Reis,Atelier Lua,Vaso de cerâmica azul,85.0,1
3,P003,2026-04-12,Marina Costa,Madeira Viva,Porta-copos rústico (kit 4),60.0,1
4,P003,2026-04-12,Marina Costa,Tecidos Nelm,Almofada bordada flores,45.0,2
5,P004,2026-04-15,Larissa Veiga,Madeira Viva,Tábua de queijo entalhada,110.0,1


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=1e994786-c624-42ec-b77d-669a6a353287' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>